In [0]:
dbutils.widgets.text("catalog","dbr_dev_ua5816bd")
dbutils.widgets.text("bronze_schema","team_crypto_bronze")
dbutils.widgets.text("silver_schema","team_crypto_silver")

CATALOG = dbutils.widgets.get("catalog")
BRONZE_SCHEMA = dbutils.widgets.get("bronze_schema")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")

BRONZE_TABLE = (f"{CATALOG}.{BRONZE_SCHEMA}.ohlc_batch")
SILVER_TABLE = (f"{CATALOG}.{SILVER_SCHEMA}.ohlc_prices")


In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

bronze_ohlc_df = spark.table(BRONZE_TABLE)

display(bronze_ohlc_df.orderBy(F.col("event_timestamp").desc()).limit(10))

In [0]:
silver_ohlc_df = (
    bronze_ohlc_df
    # standardize symbol
    .withColumn(
        "symbol",
        F.regexp_replace(
            F.upper(F.col("symbol")),
            "USD$",
            ""
        )
    )
    # usd classification
    .withColumn(
        "quote_currency",
        F.lit("USD")
    )

    # standartize timestamp
    .withColumnRenamed(
        "event_timestamp",
        "event_time"
    )
)

In [0]:
silver_clean_df = (
    silver_ohlc_df

    # min check
    .filter(F.col("symbol").isNotNull() & F.col("event_time").isNotNull())
    .dropDuplicates(["symbol", "event_time"])
)

print("Rows prepared for Silver:", silver_clean_df.count())

In [0]:
silver_clean_df = silver_clean_df.select(
    "symbol",
    "quote_currency",
    "event_time",
    "timestamp_unix",
    "open",
    "high",
    "low",
    "close",
    "vwap",
    "volume",
    "trade_count",
    "source",
    "source_filename",
    "ingestion_timestamp",
    "load_date"
)

silver_clean_df.printSchema()

display(
    silver_clean_df
    .orderBy(F.col("event_time").desc())
    .limit(10)
)

In [0]:
if spark.catalog.tableExists(SILVER_TABLE):

    before_count = spark.table(SILVER_TABLE).count()

    silver_delta = DeltaTable.forName(spark, SILVER_TABLE)

    (
        silver_delta.alias("target")
        .merge(
            silver_clean_df.alias("source"),
            """
            target.symbol = source.symbol
            AND target.event_time = source.event_time
            """
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    action = "Silver MERGE completed"

else:

    before_count = 0

    (
        silver_clean_df.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(SILVER_TABLE)
    )

    action = "Silver table created"

after_count = spark.table(SILVER_TABLE).count()

print(action)
print("Rows before:", before_count)
print("Rows after:", after_count)
print("New rows inserted:", after_count - before_count)

In [0]:
silver_result_df = spark.table(SILVER_TABLE)

duplicate_count = (
    silver_result_df
    .groupBy("symbol", "event_time")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Total Silver rows:", silver_result_df.count())
print("Duplicate keys:", duplicate_count)

assert duplicate_count == 0, "Duplicates detected in Silver"

display(
    silver_result_df
    .groupBy("symbol")
    .agg(
        F.count("*").alias("records"),
        F.min("event_time").alias("first_event"),
        F.max("event_time").alias("last_event"),
        F.min("close").alias("minimum_price"),
        F.max("close").alias("maximum_price")
    )
    .orderBy("symbol")
)